In [ ]:
import pandas as pd
import glob
import os
from scipy import stats
from itertools import combinations
from functions import *

#### Demographics (Table 1)

In [ ]:
all_CPET_files = sorted(glob.glob('data/CPET/*.csv'))
all_NIRS_files = sorted(glob.glob('data/NIRS/*.csv'))
participant_info = pd.read_csv('data/metadata/participant_info.csv')
id_to_diag = dict(zip(participant_info['participant_id'], participant_info['clinical_diagnosis']))
pul_rulm_df = pd.read_csv('data/metadata/pul_rulm.csv')
rev_df = pd.read_csv('data/metadata/a6mct_revolutions.csv')

cpet_demographics = pd.read_csv('data/metadata/cpet_demographics.csv')
cpet_demographics['height'] = cpet_demographics['height'] / 100

cpet_basenames = [os.path.basename(f).replace('.csv', '') for f in all_CPET_files]
cpet_info_df = pd.DataFrame({'participant_id': [name.split('_')[0] for name in cpet_basenames], 'filename': cpet_basenames})

cpet_demographics = cpet_info_df.merge(cpet_demographics, on='participant_id', how='left', validate='1:1')
assert len(cpet_demographics) == len(cpet_info_df), 'merge changed row count'
assert cpet_demographics['height'].notna().all(), 'some CPET files have no demographics'
cpet_demographics['bmi'] = cpet_demographics['weight'] / (cpet_demographics['height'] ** 2)
cpet_demographics = cpet_demographics.merge(participant_info[['participant_id', 'race', 'clinical_diagnosis', 'fx_status_sma', 'fx_status_all']], on='participant_id', how='left')
cpet_demographics['sex'] = cpet_demographics['sex'].map({'M': 'Male', 'F': 'Female'})

cpet_demographics = attach_nearest_test(cpet_demographics, pul_rulm_df, ['pul_total'], only_dx='DMD')
cpet_demographics = attach_nearest_test(cpet_demographics, pul_rulm_df, ['rulm_best'], only_dx='SMA')
for diag, col in [('DMD', 'pul_total'), ('SMA', 'rulm_best')]:
    missing = cpet_demographics[(cpet_demographics['clinical_diagnosis'] == diag) & cpet_demographics[col].isna()]
    for _, r in missing.iterrows():
        print(f"{r['participant_id']} ({diag}): no {col.split('_')[0].upper()} within 7 days of CPET")

In [ ]:
demo_info = get_patient_info(cpet_demographics)
demo_info.to_csv('outputs/tables/demographics_table1.csv', index=False)
demo_info

#### Peak CPET (Table 2)

In [ ]:
x_sectional_peak_table, peak_cpet_data = make_peak_cpet_table(all_CPET_files, cpet_demographics, rev_df, return_data=True)
x_sectional_peak_table.to_csv('outputs/tables/cross_sectional_peak_table.csv')
x_sectional_peak_table

#### CPET recovery kinetics (Supplementary Table 6)

In [ ]:
x_sectional_recovery_table = make_recovery_table(all_CPET_files, cpet_demographics, min_sustain_sec=10)
x_sectional_recovery_table.to_csv('outputs/tables/cross_sectional_recovery_table.csv')
x_sectional_recovery_table

#### NIRS cohort and optode QC (Supplementary Tables 3, 4)

In [ ]:
clean_optodes_marked = pd.read_csv('data/metadata/nirs_optode_selection.csv')
analysis_basenames = {os.path.basename(f).replace('.csv', '') for f in all_CPET_files}
qc_df = clean_optodes_marked.merge(cpet_demographics[['filename', 'bmi']], left_on='basename', right_on='filename', how='left').drop(columns=['filename'])
qc_df = qc_df[qc_df['basename'].isin(analysis_basenames)].sort_values('basename').reset_index(drop=True)

tri_cols, bi_cols = [f'tri_optode_{i}' for i in (1, 2, 3)], [f'bi_optode_{i}' for i in (1, 2, 3)]
excluded_one_muscle = qc_df[qc_df[tri_cols].eq(False).all(axis=1) ^ qc_df[bi_cols].eq(False).all(axis=1)].reset_index(drop=True)
included_at_least_one_muscle = qc_df[qc_df[tri_cols].eq(True).any(axis=1) | qc_df[bi_cols].eq(True).any(axis=1)].reset_index(drop=True)

excluded_one_muscle['participant_id'] = excluded_one_muscle['basename'].str.split('_').str[0]
excluded_one_muscle = excluded_one_muscle.merge(rev_df[['participant_id', 'mct_cum6']], on='participant_id', how='left').rename(columns={'mct_cum6': 'Total # of revolutions'})
excluded_one_muscle['Total # of revolutions'] = excluded_one_muscle['Total # of revolutions'].astype('Int64')
excluded_one_muscle = excluded_one_muscle.drop(columns=['participant_id'])
excluded_one_muscle['diagnosis'] = pd.Categorical(excluded_one_muscle['diagnosis'], categories=['Control', 'SMA', 'DMD'], ordered=True)
excluded_one_muscle = excluded_one_muscle.sort_values(by=['diagnosis']).reset_index(drop=True)
excluded_one_muscle.to_csv('outputs/tables/supp_table4.csv', index=True)

In [ ]:
supp_table3 = build_supp_table3(included_at_least_one_muscle)
supp_table3.to_csv('outputs/tables/supp_table3.csv', index=True)
supp_table3

#### NIRS trajectories and metrics (Figure 2, Table 3)

In [ ]:
all_NIRS_dfs = load_nirs_dfs(all_NIRS_files, participant_info)
cpet_nirs_dfs = build_cpet_nirs_dfs(all_CPET_files, all_NIRS_files, clean_optodes_marked)
plot_combined_trajectories_qc(all_NIRS_dfs, clean_optodes_marked)

In [ ]:
nirs_table, nirs_long = create_nirs_combined_table(all_NIRS_dfs, cpet_nirs_dfs, id_to_diag, clean_optodes_marked, return_long=True)
nirs_table.to_csv('outputs/tables/nirs_table3.csv', index=False)
nirs_table

#### Adipose tissue thickness sensitivity analysis (Supplementary Table 5)

In [ ]:
parts = []
for m, cal_col in {'tri': 'triceps_caliper', 'bi': 'biceps_caliper'}.items():
    optodes = [f'{m}_optode_{i}' for i in (1, 2, 3)]
    d = included_at_least_one_muscle[['basename', 'diagnosis', 'bmi', cal_col] + optodes].copy()
    d['muscle'] = m
    d['caliper'] = d.pop(cal_col)
    d['n_optodes_kept'] = d[optodes].eq(True).sum(axis=1)
    parts.append(d.drop(columns=optodes))
by_muscle = pd.concat(parts, ignore_index=True)

attenuated_but_not_excluded = by_muscle[(by_muscle['caliper'] > 3.0) & (by_muscle['n_optodes_kept'] > 0)]
print(len(attenuated_but_not_excluded), 'muscle recordings from', attenuated_but_not_excluded['basename'].nunique(), 'participants')
print(attenuated_but_not_excluded.groupby(['diagnosis', 'muscle']).size())

In [ ]:
clean_optodes_sens = apply_caliper_exclusion(clean_optodes_marked)
cpet_nirs_dfs_sens = build_cpet_nirs_dfs(all_CPET_files, all_NIRS_files, clean_optodes_sens)
nirs_table_sens, nirs_long_sens = create_nirs_combined_table(all_NIRS_dfs, cpet_nirs_dfs_sens, id_to_diag, clean_optodes_sens, return_long=True)
nirs_table_sens.to_csv('outputs/tables/nirs_sensitivity.csv', index=False)

cmp = nirs_table.merge(nirs_table_sens, on='Metric', suffixes=('_primary', '_sens'))
for c in ['Control vs. SMA', 'Control vs. DMD', 'SMA vs. DMD']:
    cmp[f'{c}_flipped'] = (cmp[f'{c}_primary'] < 0.05) != (cmp[f'{c}_sens'] < 0.05)
print('Comparisons changed:', int(cmp.filter(like='_flipped').to_numpy().sum()), 'of', cmp.filter(like='_flipped').size)

def tri_recovery_slope(long, dx):
    return long[(long['muscle'] == 'tri') & (long['metric'] == 'tsi_recoveryslope') & (long['diagnosis'] == dx)]['value'].dropna()

for name, long in [('primary', nirs_long), ('sensitivity', nirs_long_sens)]:
    print(f"Cliff's delta, triceps TSI recovery slope, Control vs SMA ({name}):", round(cliffs_delta(tri_recovery_slope(long, 'Control'), tri_recovery_slope(long, 'SMA')), 2))
nirs_table_sens

#### Correlations with upper-limb function (Supplementary Table 7)

In [ ]:
function_corr_table = make_function_corr_table(peak_cpet_data, nirs_long, cpet_demographics)
function_corr_table.to_csv('outputs/tables/function_correlations.csv')
function_corr_table

In [ ]:
rulm = cpet_demographics.loc[cpet_demographics['clinical_diagnosis'] == 'SMA', 'rulm_best']
print(f'SMA RULM: {(rulm < 10).sum()} below 10, {(rulm > 35).sum()} above 35, of {rulm.notna().sum()}')

#### Δ[HHb]/VO₂ ratio (Supplementary Tables 8, 9; Supplementary Figure 2)

In [ ]:
hhb_vo2_table = build_hhb_vo2_table(cpet_nirs_dfs, id_to_diag)
hhb_vo2_summary = build_hhb_vo2_summary_table(hhb_vo2_table)
hhb_vo2_summary

In [ ]:
crossing_tri = build_ss_crossing_table(cpet_nirs_dfs, id_to_diag, muscle='tri')
crossing_bi = build_ss_crossing_table(cpet_nirs_dfs, id_to_diag, muscle='bi')
pd.concat({'Triceps': summarize_ss_crossing(crossing_tri), 'Biceps': summarize_ss_crossing(crossing_bi)})

In [ ]:
group_order = ['Control', 'SMA', 'DMD']
for muscle in ['tri', 'bi']:
    diagnostics_by_group = {g: get_range_diagnostics(cpet_nirs_dfs, id_to_diag, g, muscle=muscle) for g in group_order}
    fig = plot_range_quadrants(diagnostics_by_group, muscle=muscle)
    fig.savefig(f'outputs/figures/hhb_vo2_range_quadrants_{muscle}.png', dpi=600, bbox_inches='tight')
    for g, d in diagnostics_by_group.items():
        print(f"{muscle} {g}: {(d['hhb_range'] < 0).sum()}/{len(d)} ({(d['hhb_range'] < 0).mean()*100:.0f}%) with negative HHb range")

In [ ]:
intensity_df = compute_a6mct_relative_intensity(cpet_nirs_dfs, peak_cpet_data, id_to_diag)
for g1, g2 in combinations(group_order, 2):
    res = []
    for col in ['a6mct_vo2_ss_L', 'a6mct_pct_peak']:
        v1 = intensity_df.loc[intensity_df['diagnosis'] == g1, col].dropna()
        v2 = intensity_df.loc[intensity_df['diagnosis'] == g2, col].dropna()
        res.append((col, len(v1), len(v2), v1.median(), v2.median(), stats.mannwhitneyu(v1, v2, alternative='two-sided')[1]))
    for (col, n1, n2, m1, m2, _), p in zip(res, bh_adjust([r[-1] for r in res])):
        print(f'{col} {g1} vs {g2}: n={n1},{n2}, medians={m1:.2f}, {m2:.2f}, FDR p={p:.3f}')